# Sales Data Cleaning
Cleans the raw movie sales export: fixes column names, recovers missing titles, fixes mis-encoded text, parses release dates, flags runtime outliers, and adds a primary key.

In [1]:
import pandas as pd
import numpy as np

## 1. Load data

In [2]:
# Load the raw Excel export
sales = pd.read_excel(r"/Users/huyennguyen/Documents/MDD/SQL/Data Files/sales.xlsx")

In [3]:
# Quick profile of the raw data: shape, dtypes, and % missing per column
print("Shape:", sales.shape)
sales.info()
print("\nMissing values (%):")
print((sales.isnull().mean() * 100).round(2))

Shape: (30612, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30612 entries, 0 to 30611
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   year                      30612 non-null  int64  
 1   release_date              30612 non-null  object 
 2   title                     30604 non-null  object 
 3   genre                     28908 non-null  object 
 4   international_box_office  21575 non-null  float64
 5   domestic_box_office       11884 non-null  float64
 6   worldwide_box_office      21575 non-null  float64
 7   production_budget         4480 non-null   float64
 8   Unnamed: 8                0 non-null      float64
 9   opening_weekend           10929 non-null  float64
 10  theatre_count             10963 non-null  float64
 11  avg run per theatre       10952 non-null  float64
 12  runtime                   24559 non-null  float64
 13  keywords                  12517 non-null  

## 2. Clean structure (columns, empty columns, whitespace)

In [4]:
# Standardize column names: trim spaces, lowercase, replace spaces with underscores
sales.columns = (
    sales.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [5]:
# Drop columns that are entirely empty (e.g. a stray "Unnamed: 8" column)
sales = sales.dropna(axis=1, how="all")

In [6]:
# Strip leading/trailing whitespace from every text (object) column
text_columns = sales.select_dtypes(include="object").columns
for column in text_columns:
    sales[column] = sales[column].str.strip()

In [7]:
# Check for fully duplicated rows
print("Duplicate rows:", sales.duplicated().sum())

Duplicate rows: 0


## 3. Recover missing titles from the URL\n~78 rows have no `title`. The movie slug in `url` (e.g. `.../movie/Bakha-Satang-(S-Korea)`) gives us the title back.

In [8]:
# Extract the title slug from the URL and turn dashes into spaces
sales["title_from_url"] = (
    sales["url"]
    .str.extract(r"/movie/([^/?#]+)", expand=False)
    .str.replace("-", " ", regex=False)
)

# Some slugs end with a "(YYYY)" year tag -> strip it off
sales["title_clean"] = (
    sales["title_from_url"]
    .str.replace(r"\s*\(\d{4}\)$", "", regex=True)
    .str.strip()
)

# Fill only the missing titles with the recovered value, then drop the helper columns
sales.loc[sales["title"].isnull(), "title"] = sales.loc[sales["title"].isnull(), "title_clean"]
sales = sales.drop(columns=["title_from_url", "title_clean"])

print("Missing titles remaining:", sales["title"].isnull().sum())

Missing titles remaining: 0


## 4. Fix mis-encoded titles (mojibake)\nA few titles were double-encoded (UTF-8 bytes misread as Latin-1), e.g. `AmÃ…Â¾inai` instead of `Amžinai`. `encoding_issue` flags every row that got fixed, for audit purposes.

In [9]:
def try_encoding(text):
    # Reverse a UTF-8 -> Latin-1 mis-decode; leave normal text untouched.
    if pd.isna(text):
        return text, False
    try:
        fixed = text.encode("latin1").decode("utf-8")
        return fixed, fixed != text
    except (UnicodeDecodeError, UnicodeEncodeError):
        return text, False

fixed_and_flag = sales["title"].apply(try_encoding)
sales["title"] = fixed_and_flag.apply(lambda x: x[0])
sales["encoding_issue"] = fixed_and_flag.apply(lambda x: x[1])

print("Titles fixed for encoding:", sales["encoding_issue"].sum())

Titles fixed for encoding: 7


In [10]:
# Normalize casing/spacing so titles compare and group consistently
sales["title"] = sales["title"].str.lower().str.strip()

## 5. Parse release dates\n`release_date` is text without a year (e.g. `January 1st`). Strip the ordinal suffix and combine with `year` to get a real date.

In [11]:
# Remove ordinal suffixes ("1st" -> "1", "2nd" -> "2", ...) so the date is parseable
sales["release_date_temp"] = sales["release_date"].str.replace(
    r"(\d+)(st|nd|rd|th)", r"\1", regex=True
)

# Combine with the year column and parse; unparseable rows become NaT instead of raising
sales["release_date_clean"] = pd.to_datetime(
    sales["release_date_temp"] + " " + sales["year"].astype(str),
    errors="coerce"
)
sales = sales.drop(columns=["release_date_temp"])

print("Rows where date could not be parsed:", sales["release_date_clean"].isnull().sum())

Rows where date could not be parsed: 46


## 6. Add a primary key

In [12]:
# Reset the index and add a stable, unique row identifier
sales = sales.reset_index(drop=True)
sales.insert(0, "sales_id", range(1, len(sales) + 1))

## 7. Validate numeric columns

In [13]:
# Sanity-check: box office / budget / runtime figures should never be negative
numeric_columns = [
    "international_box_office", "domestic_box_office", "worldwide_box_office",
    "production_budget", "opening_weekend", "theatre_count",
    "avg_run_per_theatre", "runtime"
]
for column in numeric_columns:
    print(column, "negative values:", (sales[column] < 0).sum())

international_box_office negative values: 0
domestic_box_office negative values: 0
worldwide_box_office negative values: 0
production_budget negative values: 0
opening_weekend negative values: 0
theatre_count negative values: 0
avg_run_per_theatre negative values: 0
runtime negative values: 0


## 8. Flag and correct runtime outliers\nRuntimes above 400 minutes are implausible for a feature film. Most are legitimate (e.g. art-house films), but 3 were confirmed as data-entry errors by checking the source page and are corrected here by `sales_id`.

In [14]:
# Flag implausible runtimes for review
sales["runtime_outlier"] = sales["runtime"] > 400

# Start from the raw runtime, then apply verified corrections
sales["runtime_clean"] = sales["runtime"]

# Verified against the source page for each title:
sales.loc[sales["sales_id"] == 27958, "runtime_clean"] = 83  # Barbara Lee: Speaking Truth to Power
sales.loc[sales["sales_id"] == 16351, "runtime_clean"] = 88  # Amžinai kartu
sales.loc[sales["sales_id"] == 3432, "runtime_clean"] = 77   # Meesterspion

# Label each row's review status for transparency
sales["runtime_status"] = "normal"
sales.loc[sales["runtime_outlier"], "runtime_status"] = "reviewed_outlier"
sales.loc[sales["runtime_clean"] != sales["runtime"], "runtime_status"] = "corrected"

## 9. Final quality check and export

In [15]:
print("FINAL DATA QUALITY CHECK")
print("Rows:", len(sales), "| Columns:", len(sales.columns))
print("Duplicate rows:", sales.duplicated().sum())
print("Unique sales_id:", sales["sales_id"].is_unique)
print("Missing titles:", sales["title"].isnull().sum())
print("Missing URLs:", sales["url"].isnull().sum())
print("Unparsed release dates:", sales["release_date_clean"].isnull().sum())
print("Runtime outliers flagged:", sales["runtime_outlier"].sum())
print("Encoding issues fixed:", sales["encoding_issue"].sum())

FINAL DATA QUALITY CHECK
Rows: 30612 | Columns: 21
Duplicate rows: 0
Unique sales_id: True
Missing titles: 0
Missing URLs: 0
Unparsed release dates: 46
Runtime outliers flagged: 7
Encoding issues fixed: 7


In [16]:
# Export the cleaned dataset
sales.to_excel("sales_cleaned_v1.xlsx", index=False)
print("Saved:", sales.shape)

Saved: (30612, 21)
